# Synset count distribution across the wn_synset vocabulary

**Primary author:** Victoria

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Exploratory analysis: for every word in the full wn_synset-scope vocabulary
(53,930 words), count its WordNet synsets and summarize the distribution.
The shape of this distribution informs the design of new phrase construction
strategies — in particular, how many words are highly polysemous (and therefore
harder to pin to a specific sense from a decontextualized phrase) versus how
many have exactly one sense.

In [ ]:
# === Setup
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.corpus import wordnet as wn

# This notebook lives at custom_embedding_model/planning/exploration/,
# so data and outputs are two directories up.
DATA_DIR = Path("../../data")
OUTPUT_DIR = Path("../../outputs")

VOCAB_PATH = DATA_DIR / "filtered_split" / "wn_synset" / "vocabulary.csv"
FIGURE_PATH = OUTPUT_DIR / "figures" / "synset_count_distribution.png"

assert VOCAB_PATH.exists(), f"Missing input: {VOCAB_PATH}"
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load the vocabulary

`vocabulary.csv` is the canonical full-scope wn_synset vocabulary: every word
that appears as a definition or answer in a clue row where both sides have at
least one WordNet synset. The `row` column is the canonical embedding index
and must never be reordered. The words are already stored in WordNet-ready
form (lowercased, multi-word entries joined with underscores).

In [ ]:
# === Load vocabulary
vocab = pd.read_csv(
    VOCAB_PATH,
    keep_default_na=False,  # "nan" (grandmother) is a valid crossword entry
    na_values=[""],
)

print(f"Loaded {len(vocab):,} vocabulary rows")
print(f"Columns: {list(vocab.columns)}")
vocab.head()

## Count synsets per word

Call `wn.synsets(word)` once per vocabulary entry and record the count. Because
every word in this vocabulary was filtered to have at least one synset, every
count should be ≥ 1 — we verify that invariant below. Runtime is a few seconds
for ~54k words on a warm WordNet cache, but we time it anyway.

In [ ]:
# === Compute synset counts
t0 = time.time()

# Words are already WordNet-ready (underscored, lowercased) so no transformation needed.
synset_counts = np.array([len(wn.synsets(w)) for w in vocab["word"]], dtype=int)

elapsed = time.time() - t0
print(f"Computed synset counts for {len(synset_counts):,} words in {elapsed:.1f}s")

# Invariant: wn_synset scope guarantees ≥ 1 synset per word.
assert (synset_counts >= 1).all(), "Found a vocabulary word with 0 synsets"

# Attach to the vocab frame for downstream analysis.
vocab["synset_count"] = synset_counts

## Descriptive statistics and bin counts

`.describe()` gives the overall shape (median, quartiles, max). The bin counts
below answer concrete design questions: what fraction of the vocabulary has
exactly one sense (no ambiguity), and how far out into the long tail the
highly polysemous words extend.

In [ ]:
# === Descriptive statistics and bin counts
counts = pd.Series(synset_counts)
print("Synset count distribution:")
print(counts.describe())
print()

n = len(counts)
# Exclusive upper-bin buckets: "exactly 1" vs. cumulative "≥ k" thresholds.
buckets = [
    ("exactly 1 synset", (counts == 1).sum()),
    ("2+ synsets",       (counts >= 2).sum()),
    ("5+ synsets",       (counts >= 5).sum()),
    ("10+ synsets",      (counts >= 10).sum()),
    ("20+ synsets",      (counts >= 20).sum()),
]
for label, k in buckets:
    print(f"  {label:20s}: {k:>7,}  ({k / n:.1%})")

## Distribution figure

Two panels side by side. The left panel shows the full range so the long tail
of highly polysemous words is visible; the right panel zooms to the 1–20 range
where the bulk of the mass sits, making the shape of the head of the
distribution legible. Both panels use integer bins with black edges.

In [ ]:
# === Two-panel distribution figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full range. Integer bins from 1 to max+1 (right edge is exclusive in np.histogram).
max_count = int(synset_counts.max())
bins_full = np.arange(1, max_count + 2)
axes[0].hist(synset_counts, bins=bins_full, edgecolor="black")
axes[0].set_xlabel("Synset count")
axes[0].set_ylabel("Number of words")
axes[0].set_title(f"Full range (1 to {max_count})")

# Right: zoom to 1-20 to show the shape of the head of the distribution.
bins_zoom = np.arange(1, 22)
axes[1].hist(np.clip(synset_counts, 1, 20), bins=bins_zoom, edgecolor="black")
axes[1].set_xlabel("Synset count")
axes[1].set_ylabel("Number of words")
axes[1].set_title("Zoom: 1–20 synsets")
axes[1].set_xticks(range(1, 21))

fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=300)
plt.show()

print(f"Saved figure to {FIGURE_PATH}")

## Sample words at key polysemy levels

The numeric distribution is easier to interpret with concrete examples. For
each of a handful of synset-count values spanning unambiguous (1) to highly
polysemous (20), sample a few vocabulary words and print them alongside the
level's total size. This gives an intuitive feel for what words sit at each
point on the distribution — the 1-synset words tend to be proper nouns or
technical terms, while the 20-synset words are typically short common verbs.

In [ ]:
# === Sample words at key polysemy levels
# Spans unambiguous (1) through the long tail (20). Skip levels with no words.
for k in [1, 2, 3, 5, 10, 20]:
    at_k = vocab[vocab["synset_count"] == k]
    if len(at_k) == 0:
        continue
    # Sample up to 8 words; random_state pinned for reproducibility.
    sample = at_k.sample(n=min(8, len(at_k)), random_state=42)
    words = sample["word"].tolist()
    print(f"synset_count = {k:>2d}  (n={len(at_k):>6,})  examples: {words}")

## Synset details for high-polysemy words

For a handful of high-polysemy words (≥ 10 synsets), print every synset
WordNet returns along with its definition. This makes concrete the problem
that decontextualized phrase construction strategies have to solve: when the
same surface word maps to many genuinely distinct senses, the choice of which
synset to use (e.g., `synsets(word)[0]`, the most frequent sense) materially
changes the meaning of the resulting phrase.

In [ ]:
# === Synset details for high-polysemy words
# Pick 5 words with ≥ 10 synsets and show every sense WordNet offers.
high_poly = vocab[vocab["synset_count"] >= 10]
sample = high_poly.sample(n=5, random_state=42)

for word in sample["word"]:
    print("=" * 72)
    print(f"{word}  (synset_count = {len(wn.synsets(word))})")
    print("=" * 72)
    # Enumerate so the reader sees the index; [0] is WordNet's most-frequent sense.
    for i, syn in enumerate(wn.synsets(word)):
        print(f"  [{i}] {syn.name():30s} {syn.definition()}")
    print()

## Interactive single-word lookup

A convenience cell for inspecting a single word's synsets, definitions, and
first usage example. Re-run after changing `WORD` to explore candidates for
phrase construction — this is the most direct way to see whether a given
word has a usable WordNet example (`f_common_wnex`) or only a definition
(`f_common_wndef`).

In [ ]:
# === Interactive single-word lookup
WORD = "plant"

synsets = wn.synsets(WORD)
print(f"{WORD}: {len(synsets)} synset(s)")
print("-" * 72)
for i, syn in enumerate(synsets):
    print(f"[{i}] {syn.name()}")
    print(f"    definition: {syn.definition()}")
    examples = syn.examples()
    # WordNet usage examples are optional; note absence explicitly so the
    # reader can see which senses would be excluded from f_common_wnex.
    if examples:
        print(f"    example:    {examples[0]}")
    else:
        print(f"    example:    (no usage example available)")
    print()

## Distribution excluding single-synset words

The full-range histogram is dominated by the 1-synset bin, which compresses
the shape of the polysemous tail. Restricting to words with two or more
synsets gives a clearer view of the distribution that actually matters for
sense-disambiguation design decisions — those are the words where choosing
a synset (e.g. the most-frequent sense) materially changes the phrase's
meaning. Same two-panel layout as before so the two figures are directly
comparable.

In [ ]:
# === Two-panel distribution figure — polysemous words only
# Restrict to words with at least 2 synsets (exclude unambiguous words).
polysemous = vocab[vocab["synset_count"] >= 2]
poly_counts = polysemous["synset_count"].to_numpy()

print(f"Polysemous subset: {len(polysemous):,} words "
      f"({len(polysemous) / len(vocab):.1%} of vocabulary)")
print()
print("Synset count distribution (polysemous words only):")
print(polysemous["synset_count"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full range of the polysemous subset. Integer bins from 2 to max+1.
max_poly = int(poly_counts.max())
bins_full = np.arange(2, max_poly + 2)
axes[0].hist(poly_counts, bins=bins_full, edgecolor="black")
axes[0].set_xlabel("Synset count")
axes[0].set_ylabel("Number of words")
axes[0].set_title(f"Full range (2 to {max_poly})")

# Right: zoom to 2-20, clipping the 20+ tail into the last bin (matches
# the pattern from the original full-vocabulary figure).
bins_zoom = np.arange(2, 22)
axes[1].hist(np.clip(poly_counts, 2, 20), bins=bins_zoom, edgecolor="black")
axes[1].set_xlabel("Synset count")
axes[1].set_ylabel("Number of words")
axes[1].set_title("Zoom: 2\u201320 synsets")
axes[1].set_xticks(range(2, 21))

fig.suptitle("Synset count distribution \u2014 polysemous words only (\u22652 synsets)")
fig.tight_layout()

POLY_FIGURE_PATH = OUTPUT_DIR / "figures" / "synset_count_distribution_polysemous.png"
fig.savefig(POLY_FIGURE_PATH, dpi=300)
plt.show()

print(f"Saved figure to {POLY_FIGURE_PATH}")

## Summary

This exploratory notebook loaded the full wn_synset-scope vocabulary
(53,930 words) and computed, for each word, the number of WordNet synsets
via `wn.synsets(word)`. It reports descriptive statistics and cumulative bin
counts at the 1 / 2+ / 5+ / 10+ / 20+ thresholds, and saves a two-panel
histogram (full range + 1–20 zoom) to
`custom_embedding_model/outputs/figures/synset_count_distribution.png` at
300 dpi. The synset-count computation runtime is printed in the cell output.

The resulting distribution is the input to decisions about new phrase
construction strategies: in particular, whether strategies that disambiguate
by sense (e.g. sense-picked WordNet definitions or examples) need to handle a
long polysemy tail, and how much of the vocabulary is already unambiguous.